# Advanced Problems with Solutions: Integers, Constructors, and Bases

**Kernel:** Python 3.13

This notebook contains advanced practice problems on Python integers, base conversion, integer constructors, prefixes, validation, custom rebasing, and edge cases.

## Problem 1 — Predict `int()` Behavior

For each expression below, predict the result or exception before running the cell.

```python
int(10.999)
int(-10.999)
int("   +1010   ", 2)
int("0b1010", 0)
int("0b1010", 2)
int("0b1010", 10)
int("1_000_000")
int("1_000", 2)
```

In [1]:
tests = [
    'int(10.999)',
    'int(-10.999)',
    'int("   +1010   ", 2)',
    'int("0b1010", 0)',
    'int("0b1010", 2)',
    'int("0b1010", 10)',
    'int("1_000_000")',
    'int("1_000", 2)',
]

for expr in tests:
    try:
        print(f"{expr:<30} -> {eval(expr)!r}")
    except Exception as ex:
        print(f"{expr:<30} -> {type(ex).__name__}: {ex}")

int(10.999)                    -> 10
int(-10.999)                   -> -10
int("   +1010   ", 2)          -> 10
int("0b1010", 0)               -> 10
int("0b1010", 2)               -> 10
int("0b1010", 10)              -> ValueError: invalid literal for int() with base 10: '0b1010'
int("1_000_000")               -> 1000000
int("1_000", 2)                -> 8


### Solution 1

`int()` truncates toward zero for real numbers.

When parsing strings:

- whitespace is allowed around the value;
- leading `+` or `-` signs are allowed;
- underscores are allowed between digits in valid positions;
- `base=0` allows Python to infer the base from prefixes such as `0b`, `0o`, and `0x`;
- if an explicit base is provided, the string must be valid in that base.

## Problem 2 — Build a Strict Base Parser

Write a function `strict_int(text, base)` that behaves like `int(text, base)` but rejects:

- leading or trailing whitespace;
- underscores;
- empty strings;
- prefixes such as `0b`, `0o`, or `0x` unless `base=0`.

The function should still allow leading `+` or `-` signs.

In [2]:
def strict_int(text: str, base: int = 10) -> int:
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    if text == "":
        raise ValueError("empty string is not allowed")

    if text != text.strip():
        raise ValueError("leading or trailing whitespace is not allowed")

    if "_" in text:
        raise ValueError("underscores are not allowed")

    if base != 0:
        unsigned = text[1:] if text[0] in "+-" else text
        lowered = unsigned.lower()
        if lowered.startswith(("0b", "0o", "0x")):
            raise ValueError("prefixes are only allowed when base=0")

    return int(text, base)


cases = [
    ("1010", 2),
    ("+1010", 2),
    ("-1010", 2),
    (" 1010", 2),
    ("1010 ", 2),
    ("1_010", 2),
    ("0b1010", 2),
    ("0b1010", 0),
]

for text, base in cases:
    try:
        print(f"strict_int({text!r}, {base}) -> {strict_int(text, base)}")
    except Exception as ex:
        print(f"strict_int({text!r}, {base}) -> {type(ex).__name__}: {ex}")

strict_int('1010', 2) -> 10
strict_int('+1010', 2) -> 10
strict_int('-1010', 2) -> -10
strict_int(' 1010', 2) -> ValueError: leading or trailing whitespace is not allowed
strict_int('1010 ', 2) -> ValueError: leading or trailing whitespace is not allowed
strict_int('1_010', 2) -> ValueError: underscores are not allowed
strict_int('0b1010', 2) -> ValueError: prefixes are only allowed when base=0
strict_int('0b1010', 0) -> 10


### Solution 2

The wrapper validates policy rules before delegating the actual conversion to Python's built-in `int()`.

This is a good practice: avoid reimplementing the full parser unless you specifically need custom parsing semantics.

## Problem 3 — Convert from Base 10 to Any Base Between 2 and 36

Implement `to_base36(n, base)` that converts a base-10 integer to a string representation in any base from 2 to 36.

Requirements:

- support negative integers;
- support zero;
- reject bases outside `2 <= base <= 36`;
- use uppercase letters `A-Z` for digits above 9.

In [3]:
def to_base36(n: int, base: int) -> str:
    digits = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

    if not isinstance(n, int):
        raise TypeError("n must be an int")

    if not isinstance(base, int):
        raise TypeError("base must be an int")

    if not 2 <= base <= 36:
        raise ValueError("base must satisfy 2 <= base <= 36")

    if n == 0:
        return "0"

    sign = "-" if n < 0 else ""
    n = abs(n)

    result = []
    while n:
        n, rem = divmod(n, base)
        result.append(digits[rem])

    return sign + "".join(reversed(result))


for value, base in [(0, 2), (10, 2), (-10, 2), (255, 16), (4095, 16), (123456789, 36)]:
    encoded = to_base36(value, base)
    decoded = int(encoded, base)
    print(f"{value:>12} in base {base:>2} -> {encoded:<10} -> decoded: {decoded}")

           0 in base  2 -> 0          -> decoded: 0
          10 in base  2 -> 1010       -> decoded: 10
         -10 in base  2 -> -1010      -> decoded: -10
         255 in base 16 -> FF         -> decoded: 255
        4095 in base 16 -> FFF        -> decoded: 4095
   123456789 in base 36 -> 21I3V9     -> decoded: 123456789


### Solution 3

Repeated division by the target base gives the digits from right to left.

For example, converting `255` to base 16 repeatedly extracts remainders:

```text
255 divmod 16 -> quotient 15, remainder 15
15  divmod 16 -> quotient 0,  remainder 15
```

The digits are therefore `F`, `F`, giving `FF`.

## Problem 4 — General Custom Alphabet Encoder and Decoder

Write two functions:

- `encode_int(n, alphabet)`
- `decode_int(text, alphabet)`

The alphabet determines the base. For example:

```python
alphabet = "01"
alphabet = "0123456789ABCDEF"
alphabet = "FT"
alphabet = "abcdefghijklmnopqrstuvwxyz"
```

Requirements:

- alphabet length must be at least 2;
- alphabet characters must be unique;
- support negative numbers;
- reject invalid characters during decoding.

In [4]:
def validate_alphabet(alphabet: str) -> None:
    if not isinstance(alphabet, str):
        raise TypeError("alphabet must be a string")

    if len(alphabet) < 2:
        raise ValueError("alphabet must contain at least two characters")

    if len(set(alphabet)) != len(alphabet):
        raise ValueError("alphabet characters must be unique")

    if "-" in alphabet or "+" in alphabet:
        raise ValueError("alphabet must not contain sign characters")


def encode_int(n: int, alphabet: str) -> str:
    validate_alphabet(alphabet)

    if not isinstance(n, int):
        raise TypeError("n must be an int")

    base = len(alphabet)

    if n == 0:
        return alphabet[0]

    sign = "-" if n < 0 else ""
    n = abs(n)
    chars = []

    while n:
        n, rem = divmod(n, base)
        chars.append(alphabet[rem])

    return sign + "".join(reversed(chars))


def decode_int(text: str, alphabet: str) -> int:
    validate_alphabet(alphabet)

    if not isinstance(text, str):
        raise TypeError("text must be a string")

    if text == "":
        raise ValueError("empty string is not a valid integer")

    sign = -1 if text.startswith("-") else 1
    body = text[1:] if text[0] in "+-" else text

    if body == "":
        raise ValueError("missing digits")

    value_by_char = {char: value for value, char in enumerate(alphabet)}
    base = len(alphabet)
    total = 0

    for char in body:
        if char not in value_by_char:
            raise ValueError(f"invalid digit {char!r} for alphabet")
        total = total * base + value_by_char[char]

    return sign * total


alphabets = [
    "01",
    "0123456789ABCDEF",
    "FT",
    "abcdefghijklmnopqrstuvwxyz",
]

for alphabet in alphabets:
    for n in [0, 1, 10, -10, 12345]:
        encoded = encode_int(n, alphabet)
        decoded = decode_int(encoded, alphabet)
        assert decoded == n
    print(f"round-trip tests passed for alphabet of length {len(alphabet)}: {alphabet!r}")

round-trip tests passed for alphabet of length 2: '01'
round-trip tests passed for alphabet of length 16: '0123456789ABCDEF'
round-trip tests passed for alphabet of length 2: 'FT'
round-trip tests passed for alphabet of length 26: 'abcdefghijklmnopqrstuvwxyz'


### Solution 4

Encoding uses repeated `divmod`.

Decoding uses positional expansion:

```text
value = value * base + digit
```

This avoids exponentiation and processes the number left to right.

## Problem 5 — Round-Trip Property Testing Without External Libraries

Test that `to_base36(n, base)` round-trips correctly through Python's built-in `int()` for many values.

Check all bases from 2 to 36 and values from `-5000` to `5000`.

In [5]:
def test_to_base36_round_trip() -> None:
    for base in range(2, 37):
        for n in range(-5000, 5001):
            encoded = to_base36(n, base)
            decoded = int(encoded, base)
            assert decoded == n, (n, base, encoded, decoded)

test_to_base36_round_trip()
print("All round-trip tests passed.")

All round-trip tests passed.


### Solution 5

A good test for base conversion is the round-trip property:

```python
int(to_base36(n, base), base) == n
```

This does not prove correctness for every possible integer, but it catches many implementation mistakes.

## Problem 6 — Decode Prefixed Integer Literals Manually

Write `parse_prefixed_integer(text)` that supports:

- binary: `0b` or `0B`
- octal: `0o` or `0O`
- hexadecimal: `0x` or `0X`
- decimal: no prefix
- optional leading `+` or `-`

Do not use `int(text, 0)` inside the function. You may use `int(body, base)` after manually detecting the base.

In [6]:
def parse_prefixed_integer(text: str) -> int:
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    text = text.strip()
    if not text:
        raise ValueError("empty string")

    sign = ""
    if text[0] in "+-":
        sign = text[0]
        text = text[1:]

    lowered = text.lower()

    if lowered.startswith("0b"):
        base = 2
        body = text[2:]
    elif lowered.startswith("0o"):
        base = 8
        body = text[2:]
    elif lowered.startswith("0x"):
        base = 16
        body = text[2:]
    else:
        base = 10
        body = text

    if body == "":
        raise ValueError("missing digits")

    return int(sign + body, base)


for text in ["101", "+101", "-101", "0b101", "-0b101", "0o77", "0xFF", "-0XfF"]:
    print(f"{text!r:<8} -> {parse_prefixed_integer(text)}")

'101'    -> 101
'+101'   -> 101
'-101'   -> -101
'0b101'  -> 5
'-0b101' -> -5
'0o77'   -> 63
'0xFF'   -> 255
'-0XfF'  -> -255


### Solution 6

The important detail is that the sign appears before the prefix:

```python
-0b101
+0xFF
```

So the sign should be removed before prefix detection.

## Problem 7 — Explain Why `bool` Is Accepted by `int()`

Investigate these expressions:

```python
int(True)
int(False)
isinstance(True, int)
True + True + 10
bin(True)
```

Then write a safer function `require_plain_int(x)` that accepts integers but rejects booleans.

In [7]:
print(int(True))
print(int(False))
print(isinstance(True, int))
print(True + True + 10)
print(bin(True))


def require_plain_int(x: int) -> int:
    if type(x) is not int:
        raise TypeError("expected a plain int, not bool or another type")
    return x


for value in [10, True, False, 3.14, "10"]:
    try:
        print(f"require_plain_int({value!r}) -> {require_plain_int(value)}")
    except Exception as ex:
        print(f"require_plain_int({value!r}) -> {type(ex).__name__}: {ex}")

1
0
True
12
0b1
require_plain_int(10) -> 10
require_plain_int(True) -> TypeError: expected a plain int, not bool or another type
require_plain_int(False) -> TypeError: expected a plain int, not bool or another type
require_plain_int(3.14) -> TypeError: expected a plain int, not bool or another type
require_plain_int('10') -> TypeError: expected a plain int, not bool or another type


### Solution 7

`bool` is a subclass of `int` in Python.

Therefore:

```python
True == 1
False == 0
isinstance(True, int) == True
```

Use `type(x) is int` instead of `isinstance(x, int)` when you specifically want to reject booleans.

## Problem 8 — Implement Base Conversion Without Strings

Write `digits_from_base10(n, base)` that returns a list of integer digits instead of a string.

Examples:

```python
digits_from_base10(10, 2)  -> [1, 0, 1, 0]
digits_from_base10(255, 16) -> [15, 15]
digits_from_base10(0, 7)    -> [0]
```

Reject negative numbers.

In [8]:
def digits_from_base10(n: int, base: int) -> list[int]:
    if type(n) is not int or type(base) is not int:
        raise TypeError("n and base must be plain integers")

    if n < 0:
        raise ValueError("n must be non-negative")

    if base < 2:
        raise ValueError("base must be at least 2")

    if n == 0:
        return [0]

    digits = []
    while n:
        n, rem = divmod(n, base)
        digits.append(rem)

    return digits[::-1]


print(digits_from_base10(10, 2))
print(digits_from_base10(255, 16))
print(digits_from_base10(0, 7))

[1, 0, 1, 0]
[15, 15]
[0]


### Solution 8

The algorithm is the same as string conversion, except each remainder is stored directly as an integer digit.

This form is useful when the digit alphabet is not known yet or when further numeric processing is needed.

## Problem 9 — Validate Digits for a Base

Write `validate_digits(digits, base)` that checks whether every digit is an integer in the range:

```python
0 <= digit < base
```

Then write `value_from_digits(digits, base)` that converts a digit list back to base 10.

In [9]:
def validate_digits(digits: list[int], base: int) -> None:
    if type(base) is not int or base < 2:
        raise ValueError("base must be an integer >= 2")

    if not digits:
        raise ValueError("digits must not be empty")

    for index, digit in enumerate(digits):
        if type(digit) is not int:
            raise TypeError(f"digit at index {index} is not a plain int")
        if not 0 <= digit < base:
            raise ValueError(f"digit {digit} at index {index} is invalid for base {base}")


def value_from_digits(digits: list[int], base: int) -> int:
    validate_digits(digits, base)

    value = 0
    for digit in digits:
        value = value * base + digit
    return value


examples = [
    ([1, 0, 1, 0], 2),
    ([15, 15], 16),
    ([0], 7),
    ([3, 2, 1], 4),
]

for digits, base in examples:
    print(f"{digits} in base {base} -> {value_from_digits(digits, base)}")

[1, 0, 1, 0] in base 2 -> 10
[15, 15] in base 16 -> 255
[0] in base 7 -> 0
[3, 2, 1] in base 4 -> 57


### Solution 9

The decoding step is a left fold:

```python
value = value * base + digit
```

For `[1, 0, 1, 0]` in base 2:

```text
(((1) * 2 + 0) * 2 + 1) * 2 + 0 = 10
```

## Problem 10 — Minimal Bytes Needed for an Integer

Write `minimal_unsigned_bytes(n)` that returns the minimum number of bytes needed to store a non-negative integer using `int.to_bytes()`.

Examples:

```python
minimal_unsigned_bytes(0)   -> 1
minimal_unsigned_bytes(255) -> 1
minimal_unsigned_bytes(256) -> 2
```

In [10]:
def minimal_unsigned_bytes(n: int) -> int:
    if type(n) is not int:
        raise TypeError("n must be a plain int")

    if n < 0:
        raise ValueError("n must be non-negative")

    return max(1, (n.bit_length() + 7) // 8)


for n in [0, 1, 255, 256, 257, 65535, 65536]:
    length = minimal_unsigned_bytes(n)
    encoded = n.to_bytes(length, "big")
    decoded = int.from_bytes(encoded, "big")
    print(f"{n:>6} -> {length} byte(s), bytes={encoded!r}, decoded={decoded}")

     0 -> 1 byte(s), bytes=b'\x00', decoded=0
     1 -> 1 byte(s), bytes=b'\x01', decoded=1
   255 -> 1 byte(s), bytes=b'\xff', decoded=255
   256 -> 2 byte(s), bytes=b'\x01\x00', decoded=256
   257 -> 2 byte(s), bytes=b'\x01\x01', decoded=257
 65535 -> 2 byte(s), bytes=b'\xff\xff', decoded=65535
 65536 -> 3 byte(s), bytes=b'\x01\x00\x00', decoded=65536


### Solution 10

`n.bit_length()` returns the number of binary bits needed to represent `n`.

Since one byte has 8 bits, the byte length is:

```python
(n.bit_length() + 7) // 8
```

The special case is zero, because `(0).bit_length()` is `0`, but storing zero still requires one byte if we want a concrete byte representation.

## Problem 11 — Signed Byte Round Trips

Write tests showing the difference between signed and unsigned byte conversion using:

```python
int.to_bytes(length, byteorder, signed=False)
int.from_bytes(data, byteorder, signed=False)
```

Use these values:

```python
[-129, -128, -1, 0, 1, 127, 128, 255]
```

Try encoding each value in one byte with `signed=True` and `signed=False`.

In [11]:
values = [-129, -128, -1, 0, 1, 127, 128, 255]

for signed in [False, True]:
    print(f"\nsigned={signed}")
    for n in values:
        try:
            data = n.to_bytes(1, "big", signed=signed)
            decoded = int.from_bytes(data, "big", signed=signed)
            print(f"{n:>4} -> {data!r:<8} -> {decoded:>4}")
        except Exception as ex:
            print(f"{n:>4} -> {type(ex).__name__}: {ex}")


signed=False
-129 -> OverflowError: can't convert negative int to unsigned
-128 -> OverflowError: can't convert negative int to unsigned
  -1 -> OverflowError: can't convert negative int to unsigned
   0 -> b'\x00'  ->    0
   1 -> b'\x01'  ->    1
 127 -> b'\x7f'  ->  127
 128 -> b'\x80'  ->  128
 255 -> b'\xff'  ->  255

signed=True
-129 -> OverflowError: int too big to convert
-128 -> b'\x80'  -> -128
  -1 -> b'\xff'  ->   -1
   0 -> b'\x00'  ->    0
   1 -> b'\x01'  ->    1
 127 -> b'\x7f'  ->  127
 128 -> OverflowError: int too big to convert
 255 -> OverflowError: int too big to convert


### Solution 11

With one byte:

- unsigned range is `0` through `255`;
- signed range is `-128` through `127`.

Values outside the selected range raise `OverflowError`.

## Problem 12 — Rebase a Large Integer and Verify It

Convert the following integer to bases 2, 8, 16, and 36 using your own function, then verify each result with `int(encoded, base)`.

```python
n = 2**256 + 2**128 + 123456789
```

In [12]:
n = 2**256 + 2**128 + 123456789

for base in [2, 8, 16, 36]:
    encoded = to_base36(n, base)
    decoded = int(encoded, base)
    print(f"base {base:>2}: length={len(encoded):>3}, round_trip={decoded == n}")
    print(encoded)
    print()

base  2: length=257, round_trip=True
10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000111010110111100110100010101

base  8: length= 86, round_trip=True
20000000000000000000000000000000000000000004000000000000000000000000000000000726746425

base 16: length= 65, round_trip=True
100000000000000000000000000000001000000000000000000000000075BCD15

base 36: length= 50, round_trip=True
6DP5QCB22IM238NR3WVP0IC7QOFHY07JM3S6KZGUDKOV4VUMMT



### Solution 12

Python integers have arbitrary precision, so this conversion works for extremely large integers.

The representation length decreases as the base increases because each digit carries more information.

## Problem 13 — Compare Built-In Format Mini-Language

Use `format()` and f-strings to represent `255` in binary, octal, decimal, and hexadecimal.

Then produce padded versions of width 8.

In [13]:
n = 255

print(format(n, "b"))
print(format(n, "o"))
print(format(n, "d"))
print(format(n, "x"))
print(format(n, "X"))

print(f"{n:08b}")
print(f"{n:08o}")
print(f"{n:08d}")
print(f"{n:08x}")
print(f"{n:08X}")

11111111
377
255
ff
FF
11111111
00000377
00000255
000000ff
000000FF


### Solution 13

The format mini-language is usually preferable for display formatting.

Common integer format codes:

- `b`: binary
- `o`: octal
- `d`: decimal
- `x`: lowercase hexadecimal
- `X`: uppercase hexadecimal

## Problem 14 — Implement a Robust `rebase()` Function

Write `rebase(text, source_base, target_base)` that converts a string from one base to another.

Requirements:

- support bases 2 through 36;
- support leading `+` and `-`;
- output uppercase letters;
- do not preserve unnecessary leading zeros;
- `rebase("0", 10, 2)` should return `"0"`.

In [14]:
def rebase(text: str, source_base: int, target_base: int) -> str:
    if not 2 <= source_base <= 36:
        raise ValueError("source_base must satisfy 2 <= source_base <= 36")

    if not 2 <= target_base <= 36:
        raise ValueError("target_base must satisfy 2 <= target_base <= 36")

    value = int(text, source_base)
    return to_base36(value, target_base)


examples = [
    ("1010", 2, 10),
    ("-1010", 2, 10),
    ("000000FF", 16, 2),
    ("123456789", 10, 36),
    ("ZZZZ", 36, 16),
    ("0", 10, 2),
]

for text, source_base, target_base in examples:
    print(f"{text!r} base {source_base} -> base {target_base}: {rebase(text, source_base, target_base)}")

'1010' base 2 -> base 10: 10
'-1010' base 2 -> base 10: -10
'000000FF' base 16 -> base 2: 11111111
'123456789' base 10 -> base 36: 21I3V9
'ZZZZ' base 36 -> base 16: 19A0FF
'0' base 10 -> base 2: 0


### Solution 14

The cleanest approach is a two-step conversion:

```text
source base string -> base 10 integer -> target base string
```

Python's `int(text, source_base)` handles the first step reliably.

## Problem 15 — Edge-Case Test Suite

Create a small test suite for the functions in this notebook.

The tests should check:

- zero conversion;
- negative conversion;
- invalid bases;
- invalid alphabets;
- invalid digits;
- round trips.

In [15]:
def assert_raises(expected_exception: type[BaseException], func, *args, **kwargs) -> None:
    try:
        func(*args, **kwargs)
    except expected_exception:
        return
    except Exception as ex:
        raise AssertionError(f"Expected {expected_exception.__name__}, got {type(ex).__name__}: {ex}") from ex
    else:
        raise AssertionError(f"Expected {expected_exception.__name__}, but no exception was raised")


def run_tests() -> None:
    assert to_base36(0, 2) == "0"
    assert to_base36(-10, 2) == "-1010"
    assert to_base36(255, 16) == "FF"

    assert_raises(ValueError, to_base36, 10, 1)
    assert_raises(ValueError, to_base36, 10, 37)

    assert encode_int(10, "01") == "1010"
    assert decode_int("1010", "01") == 10
    assert decode_int("-1010", "01") == -10

    assert_raises(ValueError, encode_int, 10, "0")
    assert_raises(ValueError, encode_int, 10, "001")
    assert_raises(ValueError, decode_int, "102", "01")

    assert digits_from_base10(0, 2) == [0]
    assert digits_from_base10(10, 2) == [1, 0, 1, 0]
    assert value_from_digits([1, 0, 1, 0], 2) == 10
    assert_raises(ValueError, value_from_digits, [2], 2)

    for base in range(2, 37):
        for n in [-1000, -1, 0, 1, 1000, 123456789]:
            assert int(to_base36(n, base), base) == n


run_tests()
print("All tests passed.")

All tests passed.


### Solution 15

Good tests include normal cases and failure cases.

A robust base-conversion implementation should be tested against:

- boundary values such as `0`, `1`, and `base - 1`;
- signs;
- invalid bases;
- invalid digits;
- round-trip identities.